In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

train = pd.read_csv("train (1).csv")
test = pd.read_csv("test (1).csv")
sample = pd.read_csv("sample_submission.csv")

target_col = [c for c in train.columns if c not in test.columns][0]
X = train.drop(columns=[target_col])
y = train[target_col]

cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
num_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

preprocess = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", "passthrough", num_cols)
])

model = LogisticRegression(max_iter=1000, n_jobs=-1)

pipe = Pipeline([
    ("prep", preprocess),
    ("model", model)
])

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

pipe.fit(X_train, y_train)

val_preds = pipe.predict_proba(X_val)[:, 1]
print("Validation ROC-AUC:", roc_auc_score(y_val, val_preds))

test_preds = pipe.predict_proba(test)[:, 1]

submission = sample.copy()
submission.iloc[:, 1] = test_preds
submission.to_csv("submission.csv", index=False)


Validation ROC-AUC: 0.9188126922890754
